In [46]:
import pandas as pd
import numpy as np

In [47]:
merged = pd.read_csv('../../data/merged/merged.csv', low_memory=False)

In [48]:
merged.shape

(54478, 140)

Drop columns with high cardinality

In [ ]:
drop_cols = [
    'price_range_mentioned',           # 692 unique — free text price ranges
    'desire_to_cancel',                # 92 unique — free text, use desire_to_cancel_clean instead
    'competitor_benefits_mentioned',   # 78 unique — free text
    'competitor_value_comparison',     # 23 unique — free text
    'agent_response_category',         # 20 unique
    'customer_renewal_response_category', # 20 unique
    'agent_renewal_pitch_category',    # 17 unique
    'customer_reaction_category',      # 13 unique
    'complaint_category',              # 22 unique
    'mentioned_competitors',           # contains raw free text rows
    'crm_membership_level',            # 11 unique with messy values like 'Band C1 (5-9 employees)'
    'call_date_x', 'call_date_y',      # date cols not used as features
    'registration_date',               # use tenure_years instead
    'renewal_month',                   # use renewal_year instead
    'closed_date',                     # admin
    'call_id', 'call_number', 'is_analysed',  # system IDs
    'prospect_outcome', 'prospect_status',    # target already in 'churn'
]

merged = merged.drop(columns=drop_cols)
merged.shape


(54478, 119)

Fixing Yes/No columns that have junk values

In [50]:
yes_no_cols = [
    'explicit_competitor_mention',
    'explicit_switching_intent',
    'price_switching_mentioned',
    'discussion_on_price_increase',
    'renewal_impact_due_to_price_increase',
    'discount_or_waiver_requested',
    'crm_financial_hardship_mentioned',
    'crm_dissatisfaction_with_support',
    'crm_refund_mentioned',
    'crm_customer_complained',
]

for col in yes_no_cols:
    if col not in merged.columns:
        continue
    merged[col] = merged[col].where(merged[col].isin(['Yes', 'No']), other='No')
    print(f"{col} unique values after cleaning: {merged[col].dropna().unique()}")

explicit_competitor_mention unique values after cleaning: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
explicit_switching_intent unique values after cleaning: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
price_switching_mentioned unique values after cleaning: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
discussion_on_price_increase unique values after cleaning: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
renewal_impact_due_to_price_increase unique values after cleaning: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
discount_or_waiver_requested unique values after cleaning: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
crm_financial_hardship_mentioned unique values after cleaning: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
crm_dissatisfaction_with_support unique values after cleaning: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
crm_refund_mentioned unique values after cleaning: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
crm_customer_compl

In [51]:
categorical_cols = merged.select_dtypes(include=['object', 'category', 'str'])
print(len(categorical_cols.columns))

80


In [ ]:
yes_no_cols = [col for col in categorical_cols.columns if set(merged[col].dropna().unique()) == {'Yes', 'No'}]
print(len(yes_no_cols))

36


In [53]:
yes_no_map = {'Yes': 1, 'No': 0}

for col in yes_no_cols:
    if merged[col].dropna().isin(['Yes', 'No']).all():
        merged[col] = merged[col].map(yes_no_map)

In [54]:
for col in yes_no_cols:
    print(f"{col} unique values after mapping: {merged[col].dropna().unique()}")

crm_customer_complained unique values after mapping: [0 1]
crm_refund_mentioned unique values after mapping: [1 0]
crm_dissatisfaction_with_support unique values after mapping: [0 1]
crm_financial_hardship_mentioned unique values after mapping: [0 1]
membership_renewal_decision unique values after mapping: [0. 1.]
serious_complaint unique values after mapping: [0. 1.]
other_complaint unique values after mapping: [0. 1.]
discussion_on_price_increase unique values after mapping: [0 1]
renewal_impact_due_to_price_increase unique values after mapping: [0 1]
discount_or_waiver_requested unique values after mapping: [0 1]
call_reschedule_request unique values after mapping: [0. 1.]
agent_flagged_membership_status_alert unique values after mapping: [0. 1.]
agent_renewal_initiation unique values after mapping: [0. 1.]
explicit_competitor_mention unique values after mapping: [0 1]
explicit_switching_intent unique values after mapping: [0 1]
price_switching_mentioned unique values after mapping: